## ▶ Colab setup — GitHub repo + dataset from Google Drive

Run this cell **first** on Google Colab. It clones the GitHub repo, installs the
`phytolabs` package, mounts your Drive, and unzips + reshapes the Leaf-rust
dataset into `data/{train,val}/{healthy,rust}` so the rest of the notebook runs
on **real data**. Edit `ZIP_PATH` if your zip lives elsewhere in Drive.

On a non-Colab machine this cell is a harmless no-op (the notebook then falls
back to the synthetic smoke-test dataset).

In [ ]:
# === Colab setup: GitHub repo + dataset from Google Drive ===================
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/Adian17/PhytoLabs.git"
ZIP_PATH = "/content/drive/MyDrive/ece186/WheatLeafRust.zip"  # <-- adjust if needed

if "google.colab" in sys.modules:
    # 1) Clone the repo + install the package.
    if not os.path.isdir("/content/PhytoLabs"):
        !git clone -q $REPO_URL /content/PhytoLabs
    %cd /content/PhytoLabs
    !pip -q install -e .
    # Editable install / _setup aren't importable mid-kernel; add paths explicitly.
    for _p in ("/content/PhytoLabs/src", "/content/PhytoLabs/notebooks"):
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 2) Mount Drive + unzip the dataset (only the first time).
    if not (Path("data/raw").exists() and any(Path("data/raw").iterdir())):
        from google.colab import drive
        drive.mount("/content/drive")
        !rm -rf data/raw && mkdir -p data/raw
        !unzip -q "$ZIP_PATH" -d data/raw

    # 3) Reshape control/diseased -> data/{train,val}/{healthy,rust} (only once).
    if not (Path("data/train/rust").exists() and any(Path("data/train/rust").glob("*"))):
        raw = Path("data/raw")
        def _find(name):
            cands = [d for d in raw.rglob("*") if d.is_dir() and d.name.lower() == name]
            if not cands:
                raise FileNotFoundError(f"No '{name}' folder under data/raw — check the zip layout.")
            train_cands = [d for d in cands if "train" in str(d).lower()]
            return str((train_cands or cands)[0])
        H, R = _find("control"), _find("diseased")
        print("healthy <-", H, "\nrust    <-", R)
        !python -m scripts.reshape_data --healthy-src "$H" --rust-src "$R" --out data --val-fraction 0.2
    print("dataset:", {c: len(list(Path("data/train", c).glob("*"))) for c in ("healthy", "rust")})

# Stage 1 — GMM + EM (unsupervised lesion segmentation)

Fit a Gaussian Mixture Model in **HSV** color space to cluster leaf pixels into **rust / green / background**, then visualize the lesion overlay (the core PhytoLabs UX).

If you don't have a real dataset yet, this notebook generates a small synthetic one so everything runs end-to-end.

In [ ]:
from _setup import DATA_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import io, segmentation, viz

data_dir = ensure_dataset()

## Load training images

In [ ]:
healthy_imgs, healthy_paths = io.load_folder(data_dir / 'train' / 'healthy')
rust_imgs, rust_paths = io.load_folder(data_dir / 'train' / 'rust')
all_imgs = healthy_imgs + rust_imgs
print(f'{len(healthy_imgs)} healthy, {len(rust_imgs)} rust training images')

plt.imshow(viz.bgr_to_rgb(rust_imgs[0])); plt.axis('off'); plt.title('example rust leaf');

## Fit the GMM

We pool subsampled HSV pixels across all training images and fit one global GMM, then label each component by its mean hue/saturation.

In [ ]:
leaf_gmm = segmentation.train_gmm(all_imgs, k=4, random_state=0)
for c in range(leaf_gmm.n_components):
    h, s, v = leaf_gmm.gmm.means_[c]
    print(f'component {c}: HSV mean=({h:.0f},{s:.0f},{v:.0f}) -> {leaf_gmm.label_map[c]}')

## Visualize segmentation + lesion overlay

Original, lesion overlay, rust mask, and green mask for a diseased leaf.

In [ ]:
bgr = rust_imgs[0]
seg = leaf_gmm.segment(bgr)
viz.plot_segmentation(bgr, seg, title='Rust leaf segmentation')
plt.show()

In [ ]:
# Same for a healthy leaf — expect (almost) no rust pixels.
bgr_h = healthy_imgs[0]
viz.plot_segmentation(bgr_h, leaf_gmm.segment(bgr_h), title='Healthy leaf segmentation')
plt.show()

## Persist the model

Saved as `artifacts/gmm.joblib` for the later stages / CLI.

In [ ]:
from _setup import ARTIFACTS_DIR
leaf_gmm.save(ARTIFACTS_DIR / 'gmm.joblib')
print('saved', ARTIFACTS_DIR / 'gmm.joblib')

> **Tuning on real data:** inspect the printed component HSV means. If rust tissue is mislabeled, adjust the hue ranges in `phytolabs.segmentation.classify_component`.